# relay_plan

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the plans. `C1B_PILOT_PLAN` (arms, cells and seeds, H23-1 to H23-8, Gate C1b and what it decides for A1 (P23-7), the pilot checks (P23-9), the report-and-wait cost rule and the cap of $250 per calendar month) was frozen on 2026-11-11 (T6), after the user approved the pilot's estimate of $2.60-2.99. `C1B_PLAN` is the pilot plan plus the pilot's verdict, the user's approval of its projection ($24.88, 2026-11-12) and the hand-recorded calls (P23-10), frozen on 2026-11-12 (T7) and pushed before the main run. Both hashes are pinned in `test_c1b_plans`. `pilotCheck23` applies P23-9's checks and `costRule20`'s report-and-wait rule.

In [ ]:
#| default_exp relay_plan

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from explRL.bridge_grid import c1Estimate
from explRL.bridge_plan import costRule20
from explRL.colony_grid import PILOT_SEED, WORLD18
from explRL.relay_grid import (ALLOW23, ARMS23, C1B_CELLS, C1B_CLEAN, NEW23, PER_BOARD23, PILOT23, REUSED23, SEEDS5, SEEDS8,
                               arm23, c1bGrid, c1bRange)

_L = 'honestPooled, N 64, independent liars, seeds 1800-1807 paired'
C1B_PILOT_PLAN = dict(
    version=1, frozen=True, plan='specs/multiagent/phase23/', model='claude-haiku-5-5', world=WORLD18, turns=40,   # frozen 2026-11-11 (T6, the user approved the $2.60-2.99 estimate)
    channel='blackboard', memory='full', effort='low', arms=ARMS23, cells=dict(
        liars=dict(C1B_CELLS), clean=dict(C1B_CLEAN), pilot=list(PILOT23),
        reused=dict(liars=dict(REUSED23['liars']), clean=dict(REUSED23['clean']), source="C1's cache ($0): c1/runs.csv, c1/reused.csv")),
    seeds=dict(eight=SEEDS8, five=SEEDS5, pilot=PILOT_SEED),
    faults=dict(independent='scripted ClueLiar in the first 16 slots (MirrorLiar: also on mirrors), guards the next 16',
                clean='no liars: 64 Claude agents, the 16 guards in the first 16 slots'),
    hypotheses=dict(
        H23_1=f'Gate C1b (i), effective: bridgeFix - bridge1, {_L}: one-sided exact sign-flip > 0, alpha 0.05 (smallest p 1/256)',
        H23_2='Gate C1b (ii), safe (non-inferiority): contamination (FR-D1: falseValues with countedFalse), bridgeFix - bridge1, '
              'the same 8 pairs: the 95th percentile of 2,000 paired bootstrap means < 0.05',
        H23_3='win-back (bridgeFix - coupled) / (squads - coupled), means over 1800-1804, > 0.5, supported only when the coupling '
              'cost is positive; truthConfirmable reported beside it',
        H23_4='filter gain (FR-D1) > 0 over the bridgeFix boards: one-sided exact sign-flip',
        H23_5='2 x 2 of evidence (no, yes) x role (solver, dedicated): bridge1, relayEvid, bridgeRole, bridgeFix on 1800-1804; '
              'each main effect is the mean of its two cell differences; descriptive',
        H23_6='bridgeFix - mirrorWindow and relayEvid - mirrorWindow, 1800-1804, descriptive',
        H23_7='clean boards: bridgeFix - squads and mirrorWindow - squads (bridge1 - squads beside), 1800-1804, descriptive',
        H23_8='coupledContamination per coupled arm, and each against coupled, 1800-1804, descriptive'),
    gate=dict(parts=['H23_1', 'H23_2'], alpha=0.05, metric='honestPooled', safety='contamination', margin=0.05, quantile=95,
              decides=dict(
                  passes="A1's bridged world has evidence: arms 1-3 a dedicated bridge in squads of 8 and solver bridges in squads "
                         "of 4; arm 4 evidence and solver bridges (P23-7, amends P19-8)",
                  noBridges='part (ii) fails: A1 trains without bridges',
                  ask='part (i) fails and part (ii) holds: stop and ask')),
    robustness='part (i) on welfareHonest; an estimate of the opposite sign marks it not robust (part ii does not depend on '
               'the welfare measure)',
    families='no Holm: the two gate parts must both hold; each secondary is reported uncorrected and labelled exploratory',
    pilot=dict(cells=list(PILOT23), seed=PILOT_SEED, analysed=False, maxInvalid=0.05, minMirrorPosts=1, minRelaysPerBridgeTurn=0.05,
               minArrivalConfirmed=1, maxBridgeTokens=1560, bridgeArms=[a for a in NEW23 if ARMS23[a].get('bridges')],
               evidenceArms=[a for a in NEW23 if ARMS23[a].get('evidence')], onFail='stop and ask',
               bridgeArmsNote="the relay check covers all three bridge arms, relayEvid's solver bridges included (the user, "
                              "2026-11-11); C1's bridge1 relayed on 0.5% of bridge turns, so relayEvid may stop the pilot to ask",
               columns=dict(mirror='mirrorPosts', relays='relaysPerBridgeTurn', arrival='arrivalConfirmed', nRelays='bridgeRelays',
                            bridgeTok='inTokPerCallBridge')),
    costRule=dict(rule='report and wait (P20-8, P23-9): the estimate before the pilot, the projection after it; the user approves each',
                  ceiling=28.0, projection="the pilot's cost per call by arm x c1bGrid's calls (1,920 per liar board, 2,560 per clean)",
                  estimate=c1bRange(), perBoard=PER_BOARD23, allowance=ALLOW23),
    cap=250.0, capNote='per calendar month (the user, 2026-11-09)', approved=dict(estimate='2026-11-11', pilot=(2.60, 2.99)),
    dirs=dict(pilot='results/multiagent/colony/c1bpilot', main='results/multiagent/colony/c1bmain',
              batches=['results/multiagent/colony/c1bmain1', 'results/multiagent/colony/c1bmain2']),
    nboot=2000, exploratory=True)
C1B_PLAN = dict(   # frozen 2026-11-12 (T7), after the pilot; nothing in the hypotheses or the gate changes
    C1B_PILOT_PLAN, frozen=True, pilotHash='e607531b64ef',
    verdict="pilotCheck23 failed 2 checks, both on relayEvid (2026-11-12): relays per bridge turn 0.003 < 0.05 (1 relay in 320 "
            "bridge turns) and no relay confirmed on arrival; every other check passed (invalid 0.14%, bridgeRole 0.141 and "
            "bridgeFix 0.066 relays per bridge turn, bridgeFix 6 relays confirmed on arrival, bridge tokens 1,081-1,172 per call). "
            "The user kept relayEvid in the main run as frozen (2026-11-12): its silence is H23-5's evidence-only cell",
    projection=dict(approved='2026-11-12', cost=24.88, ceiling=28.0, boards=33, calls=69760, pilotCost=2.73),
    calls=dict(    # P23-10, by hand before the main run; pilot-only: B3's rule (P22-16, P22-17), the pilot's cells, else C1's cached rows, unpaired
        basis='pilot seed 1890 (bridgeFix); bridge1, coupled and squads from C1 (c1/runs.csv, c1/reused.csv, c1pilot), all seeds',
        pilotOnly=dict(H23_1=dict(call='fails', estimate=-0.0125), H23_2=dict(call='holds', estimate=-0.0432),
                       H23_3=dict(call='not supported', estimate=0.115), H23_4=dict(call='supported', estimate=0.0476),
                       gate='part (i) fails and part (ii) holds: stop and ask'),
        everyDesignHelps=dict(H23_1=dict(call='passes'), H23_2=dict(call='holds'), H23_3=dict(call='supported'),
                              H23_4=dict(call='supported'), gate='passes')))

In [ ]:
#| export
def pilotCheck23(rows, plan=None):
    """P23-9 on the pilot's rows (levels '<arm>|independent'): invalid replies under 5%; mirror posts above 0; on each bridge arm,
    relays per bridge turn at least 0.05 and a bridge's mean input tokens per call at most 1,560 (P23-9 stops above it); mirror
    posts on every board (a window board must show something); on each evidence arm, at least
    one relay confirmed on arrival. Then the projection of c1bGrid at the pilot's cost per call by arm and costRule20 (always
    'ask'). A missing column fails its check. -> dict(ok, reasons, values, perCall, estimate, verdict, projected, ...)."""
    plan = plan or C1B_PILOT_PLAN
    P, col = plan['pilot'], plan['pilot']['columns']
    f = lambda r, k: float(r[k]) if r.get(k) not in (None, '') else float('nan')
    by = {arm23(r): r for r in rows}
    calls = sum(f(r, 'calls') for r in rows) or 1.0
    br, ev = [a for a in P['bridgeArms'] if a in by], [a for a in P['evidenceArms'] if a in by]
    vals = dict(invalid=sum(f(r, 'invalid') for r in rows) / calls, mirrorPosts=sum(f(r, col['mirror']) for r in rows),
                relaysPerBridgeTurn={a: f(by[a], col['relays']) for a in br}, bridgeTokens={a: f(by[a], col['bridgeTok']) for a in br},
                confirmedOnArrival={a: round(f(by[a], col['arrival']) * f(by[a], col['nRelays']), 6) for a in ev})   # 1/49 * 49 < 1
    reasons = [m for m, bad in [
        (f"missing pilot boards {sorted(set(P['cells']) - set(by))}", set(P['cells']) - set(by)),
        (f"invalid {vals['invalid']:.3f} >= {P['maxInvalid']}", not vals['invalid'] < P['maxInvalid']),
        (f"no mirror posts ({vals['mirrorPosts']})", not vals['mirrorPosts'] >= P['minMirrorPosts'])] +
        [(f'{a}: no mirror posts', not f(r, col['mirror']) >= P['minMirrorPosts']) for a, r in by.items()] +
        [(f"{a}: relays per bridge turn {v:.3f} < {P['minRelaysPerBridgeTurn']}", not v >= P['minRelaysPerBridgeTurn'])
         for a, v in vals['relaysPerBridgeTurn'].items()] +
        [(f"{a}: bridge input tokens per call {v:.0f} > {P['maxBridgeTokens']}", not v <= P['maxBridgeTokens'])
         for a, v in vals['bridgeTokens'].items()] +
        [(f'{a}: no relay confirmed on arrival ({v})', not v >= P['minArrivalConfirmed']) for a, v in vals['confirmedOnArrival'].items()] if bad]
    perCall = {a: f(r, 'cost') / f(r, 'calls') for a, r in by.items() if f(r, 'calls')}
    est = c1Estimate(c1bGrid(), perCall) if perCall else dict(cost=float('nan'))
    rule = costRule20(est['cost'], dict(costRule=plan['costRule']))
    return dict(ok=not reasons, reasons=reasons, values=vals, perCall=perCall, estimate=est, **rule)

In [ ]:
# test_c1b_plans (FR-C3, FR-C4, P23-9): both plans record the cap and its note, the cells and the gate; each is frozen
# after its approval (C1B_PILOT_PLAN at T6, C1B_PLAN at T7, hashes pinned); pilotCheck23 passes good pilot rows and names each failure, and its verdict is always 'ask'
import copy
from explRL.coop_grid import coopHash
for p in (C1B_PILOT_PLAN, C1B_PLAN):
    assert p['cap'] == 250.0 and 'calendar month' in p['capNote'] and p['gate']['margin'] == 0.05 and p['gate']['alpha'] == 0.05
    assert set(p['hypotheses']) == {f'H23_{i}' for i in range(1, 9)} and set(p['gate']['decides']) == {'passes', 'noBridges', 'ask'}
assert C1B_PILOT_PLAN['frozen'] and C1B_PLAN['frozen'] and C1B_PLAN['pilotHash'] == coopHash(C1B_PILOT_PLAN)
assert coopHash(C1B_PLAN) == '86397febb712', coopHash(C1B_PLAN)   # the frozen main plan (T7, pushed before the main run)
assert C1B_PLAN['projection']['approved'] == '2026-11-12' and C1B_PLAN['projection']['cost'] <= C1B_PLAN['costRule']['ceiling']
assert all({f'H23_{i}' for i in range(1, 5)} <= set(C1B_PLAN['calls'][k]) for k in ('pilotOnly', 'everyDesignHelps'))
assert coopHash(C1B_PILOT_PLAN) == 'e607531b64ef', coopHash(C1B_PILOT_PLAN)   # the frozen pilot plan (T6, pushed before the pilot)
assert C1B_PILOT_PLAN['approved'] == dict(estimate='2026-11-11', pilot=(2.60, 2.99))
assert C1B_PILOT_PLAN['pilot']['bridgeArms'] == ['relayEvid', 'bridgeRole', 'bridgeFix']
assert C1B_PILOT_PLAN['pilot']['evidenceArms'] == ['relayEvid', 'bridgeFix'] and C1B_PILOT_PLAN['dirs']['main'].endswith('c1bmain')
assert isinstance(coopHash(C1B_PILOT_PLAN), str)
good = lambda a: dict(level=f'{a}|independent', calls=1920.0, invalid=2.0, cost=0.66, mirrorPosts=150.0,
                      relaysPerBridgeTurn=0.1 if ARMS23[a].get('bridges') else float('nan'), bridgeRelays=30.0,
                      arrivalConfirmed=0.5, inTokPerCallBridge=1100.0)
rows = [good(a) for a in PILOT23]
P = pilotCheck23(rows)
assert P['ok'] and P['reasons'] == [] and P['verdict'] == 'ask' and abs(P['projected'] - 33 * 0.66 * (23 * 1920 + 10 * 2560) / 33 / 1920) < 1e-6
for change, want in ((dict(invalid=500.0), 'invalid'), (dict(relaysPerBridgeTurn=0.01), 'relays per bridge turn'),
                     (dict(arrivalConfirmed=0.0), 'confirmed on arrival'), (dict(inTokPerCallBridge=1600.0), 'tokens')):
    bad = [dict(r, **change) if r['level'].startswith('bridgeFix') else r for r in rows]
    Q = pilotCheck23(bad)
    assert not Q['ok'] and len(Q['reasons']) == 1 and want in Q['reasons'][0], (change, Q['reasons'])
Q = pilotCheck23([dict(r, mirrorPosts=0.0) if r['level'].startswith('mirrorWindow') else r for r in rows])
assert not Q['ok'] and Q['reasons'] == ['mirrorWindow: no mirror posts'], Q['reasons']
Q = pilotCheck23([dict(r, mirrorPosts=0.0) for r in rows[1:]])
assert not Q['ok'] and any('missing' in m for m in Q['reasons']) and any('mirror' in m for m in Q['reasons'])
assert not pilotCheck23([{k: v for k, v in r.items() if k != 'arrivalConfirmed'} for r in rows])['ok']   # a missing column fails
for n in (49, 98, 103, 107, 161, 197):                                # one confirmed relay in n is one, not 0.999...
    assert pilotCheck23([dict(r, arrivalConfirmed=1 / n, bridgeRelays=float(n)) for r in rows])['ok'], n

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()